# DiCE regression with CELIA on a NodeML pipeline

**DiCE** (Diverse Counterfactual Explanations, Mothilal et al., 2020) searches for inputs that are close to a sample, that are different from each other, and whose prediction is in a target range. For a regression model, the question is:

> *Which small changes to this part move its predicted hardness into the range 1 480 to 1 520 HV10?*

[CELIA](https://github.com/serval-uni-lu/celia) wraps DiCE for regression as `DiceRegressorExplainer`. This notebook shows two ways to use it with a trained NodeML pipeline:

* **Part A** uses the CELIA API directly, to show how `DiceRegressorExplainer` works.
* **Part B** uses the NodeML `CounterfactualEvaluator`, which does the CELIA steps for you, runs many samples in parallel with Ray, and scores every counterfactual. Use Part B for real work.

Install the optional dependencies first:

```bash
uv sync --extra counterfactuals        # or: pip install "nodeml[counterfactuals]"
```

In [ ]:
import numpy as np
import pandas as pd

from nodeml import NODE_REGISTRY
from nodeml.components.nodes.data_sources.inputs_passthrough import (
    InputsPassthroughConfig,
)
from nodeml.core.common.data.data import (
    ArrayLikeEnum,
    CategoricalData,
    DataCategoryEnum,
    DataStructureEnum,
    NumericalData,
    TabularDataContext,
)
from nodeml.core.common.enums import NodeExecutionMode
from nodeml.core.nodes.node import Port
from nodeml.core.pipeline.pipeline import Edge, Pipeline
from nodeml.core.pipeline.runners.smart_runner import SmartRunner, SmartRunnerConfig

pd.set_option("display.precision", 3)

## 1. Train a pipeline

The data is synthetic. Each row is a sintered part: its cobalt content, grain size, sintering temperature, pressing pressure, and powder grade (a categorical column). The target is the hardness `HV10`.

The pipeline splits the numerical and the categorical columns, scales the first, one-hot encodes the second, and trains a random forest.

In [ ]:
rng = np.random.default_rng(42)
n = 500
X = pd.DataFrame(
    {
        "cobalt_pct": rng.uniform(6.0, 15.0, n).round(2),
        "grain_size_um": rng.uniform(0.5, 3.0, n).round(2),
        "sinter_temp_c": rng.uniform(1380.0, 1480.0, n).round(0),
        "press_mpa": rng.uniform(100.0, 200.0, n).round(0),
        "grade": rng.choice(["A", "B", "C"], n),
    }
)
grade_offset = X["grade"].map({"A": 40.0, "B": 0.0, "C": -40.0})
y = pd.DataFrame(
    {
        "HV10": (
            1900.0
            - 35.0 * X["cobalt_pct"]
            - 110.0 * X["grain_size_um"]
            + 0.4 * (X["sinter_temp_c"] - 1430.0)
            + 0.2 * X["press_mpa"]
            + grade_offset
            + rng.normal(0.0, 5.0, n)
        ).round(1)
    }
)
X_ctx = TabularDataContext(
    list(X.columns), list(X.dtypes), [NumericalData] * 4 + [CategoricalData]
)
y_ctx = TabularDataContext(["HV10"], list(y.dtypes), [NumericalData])
X.head()

In [ ]:
source_config = InputsPassthroughConfig(
    out_ports={
        "X": Port(
            arr_type=ArrayLikeEnum.PANDAS,
            data_structure=DataStructureEnum.TABULAR,
            data_category=DataCategoryEnum.MIXED,
            data_shape="batch feature",
            desc="Process and powder features.",
        ),
        "y": Port(
            arr_type=ArrayLikeEnum.PANDAS,
            data_structure=DataStructureEnum.TABULAR,
            data_category=DataCategoryEnum.NUMERICAL,
            data_shape="batch 1",
            desc="Hardness HV10.",
            mode=[NodeExecutionMode.TRAINING, NodeExecutionMode.EVALUATION],
        ),
    }
)
model_config = NODE_REGISTRY.get_node_config_class("RandomForestRegressor")()
model_config.running_config.random_state = 0

pipe = Pipeline()
pipe.add_node("source", "InputsPassthrough", source_config)
pipe.add_node("split", "DataCategoryFilter")
pipe.add_node("scaler", "StandardScaler")
pipe.add_node("onehot", "OneHotEncoding")
pipe.add_node("concat", "FeatureConcatenate")
pipe.add_node("model", "RandomForestRegressor", model_config)
pipe.add_node("sink", "Sink")
pipe.add_edge(Edge(source="source", target="split", ports_map=[("X", "input")]))
pipe.add_edge(Edge(source="split", target="scaler", ports_map=[("numerical", "input")]))
pipe.add_edge(
    Edge(source="split", target="onehot", ports_map=[("categorical", "input")])
)
pipe.add_edge(Edge(source="scaler", target="concat", ports_map=[("output", "input_1")]))
pipe.add_edge(Edge(source="onehot", target="concat", ports_map=[("output", "input_2")]))
pipe.add_edge(Edge(source="concat", target="model", ports_map=[("output", "X")]))
pipe.add_edge(Edge(source="source", target="model", ports_map=[("y", "y")]))
pipe.add_edge(Edge(source="model", target="sink", ports_map=[("pred", "pred")]))
pipe.compile()

runner = SmartRunner(pipe, config=SmartRunnerConfig(verbose=False))
runner.train(input_data={"source": {"X": (X, X_ctx), "y": (y, y_ctx)}})

# Part A: the CELIA API

### A.1 Import CELIA

BugDoc, a dependency of CELIA, sets the root logger to `DEBUG` when it is imported. After a plain `import celia`, every debug message of every library goes to the notebook. `import_celia()` imports CELIA and restores the logging configuration.

In [ ]:
from nodeml.core.pipeline.counterfactuals.celia_adapter import import_celia

celia = import_celia()
celia.__version__

### A.2 Wrap the pipeline as a CELIA model

CELIA explains any object that implements `celia.BaseModel`: a `predict` method that takes a DataFrame of feature rows and returns one prediction for each row. Here, `predict` runs the trained pipeline in inference mode.

DiCE can return the columns in another order, so `predict` selects them in the training order first.

In [ ]:
FEATURES = list(X.columns)


class PipelineRegressor(celia.BaseModel):
    """A CELIA model that predicts with a trained NodeML pipeline."""

    def predict(self, x: pd.DataFrame) -> np.ndarray:
        rows = x[FEATURES].reset_index(drop=True)
        context = X_ctx.aligned_to(rows)
        prediction, _ = self.model.infer(input_data={"source": {"X": (rows, context)}})[
            "pred"
        ]
        return prediction.to_numpy()[:, 0]

    def predict_proba(self, x: pd.DataFrame) -> np.ndarray:
        msg = "This is a regression model."
        raise NotImplementedError(msg)


model = PipelineRegressor(runner)
model.predict(X.head(3))

### A.3 Describe the data

`celia.Data` holds the reference data (DiCE uses it to learn the range of each feature) and the constraints:

* `continuous_column_names` / `categorical_column_names`: the feature types.
* `immutable_column_names`: DiCE never changes them. Here the grade of the powder is fixed.
* `feasible_values`: a `(min, max)` tuple for a continuous column, or a list of allowed values for a categorical column. Here the furnace allows 1 400 to 1 470 °C.

The targets are the **predictions of the pipeline**, not the measured hardness: the counterfactuals explain the model.

In [ ]:
data = celia.Data(
    data=X,
    targets=pd.Series(model.predict(X), name="HV10"),
    target_name="HV10",
    continuous_column_names=[
        "cobalt_pct",
        "grain_size_um",
        "sinter_temp_c",
        "press_mpa",
    ],
    categorical_column_names=["grade"],
    immutable_column_names=["grade"],
    feasible_values={"sinter_temp_c": (1400.0, 1470.0)},
)

### A.4 Generate counterfactuals

`DiceRegressorExplainer(model, data, method=...)` builds the DiCE explainer. `generate_counterfactuals(sample, target_range, total_CFs)` returns one `Counterfactual` object for each sample row.

The DiCE version that CELIA pins has a bug in the `"random"` method: it converts every value to `float`, so it fails on string categories such as `grade`. We use the `"genetic"` method here. Part B shows how the NodeML evaluator makes `"random"` work too.

In [ ]:
sample = X.iloc[[0]]
baseline = float(model.predict(sample)[0])
target_range = [baseline + 40.0, baseline + 80.0]
print(
    f"Baseline prediction: {baseline:.1f} HV10, target range: {target_range[0]:.1f} to {target_range[1]:.1f}"
)

explainer = celia.DiceRegressorExplainer(model=model, data=data, method="genetic")
counterfactuals = explainer.generate_counterfactuals(
    sample, target_range=target_range, total_CFs=3
)
result = counterfactuals[0]
result.highlighted_counterfactuals

`highlighted_counterfactuals` shows only the changed values; `"-"` means unchanged. The `Prediction` column shows the baseline and the new prediction. The same data is in `result.counterfactuals` (the full rows) and `result.counterfactual_prediction`.

CELIA stops here. It does not check that each prediction is really in the range, it does not measure distances, and a failure (for example no counterfactual found) raises an exception.

In [ ]:
result.counterfactuals.assign(prediction=result.counterfactual_prediction)

# Part B: the NodeML `CounterfactualEvaluator`

The evaluator does the Part A steps for you, for many scenarios and samples:

1. It rebuilds the pipeline from its config and trained parameters, and inserts a `PerturbationNode` right after the data source. Each DiCE prediction then runs only the part of the pipeline after the source.
2. It builds `celia.Data` from the reference rows, the column types of the pipeline context, and your constraints.
3. It gives integer codes to string categories when the method needs them, so DiCE `"random"` works. It also handles missing values.
4. It runs each (scenario, sample) pair as a Ray task.
5. It predicts every counterfactual again with the pipeline, and scores it: validity, distance, number of changed columns, and constraint violations.

The original pipeline does not change.

In [ ]:
from nodeml.core.pipeline.counterfactuals import (
    CounterfactualEvaluator,
    CounterfactualEvaluatorConfig,
    CounterfactualScenario,
    FeatureConstraints,
    TargetRange,
)

evaluator = CounterfactualEvaluator.from_pipeline(
    pipe,
    config=CounterfactualEvaluatorConfig(
        use_ray=True,
        constraints=FeatureConstraints(
            immutable_columns=["grade"],
            feasible_ranges={"sinter_temp_c": (1400.0, 1470.0)},
        ),
    ),
)
evaluator.points

### B.1 The target range

`TargetRange` gives the target for each sample from its baseline prediction `p`:

| `kind` | Range |
|---|---|
| `"absolute"` | `[low, high]` |
| `"delta"` | `[p + low, p + high]` |
| `"relative"` | `[p * (1 + low), p * (1 + high)]` |

We ask for 40 to 80 HV10 more than the baseline, as in Part A.

In [ ]:
target = TargetRange(kind="delta", low=40.0, high=80.0)
target.resolve(1450.0)

### B.2 The three DiCE methods

A scenario is one DiCE method with its options. `explainer_kwargs` go to `DiceRegressorExplainer(...)` and `generate_kwargs` go to `generate_counterfactuals(...)`.

| Method | How it searches | Main `generate_kwargs` |
|---|---|---|
| `"random"` | Changes one random feature, then two, and so on. Fast, sparse. | `sample_size` (1000), `random_seed`, `posthoc_sparsity_param` (0.1) |
| `"genetic"` | Evolves a population and minimises a loss: distance to the target range, proximity, sparsity and diversity. | `proximity_weight` (0.2), `sparsity_weight` (0.2), `diversity_weight` (5.0), `maxiterations` (500) |
| `"kdtree"` | Returns the nearest reference rows that reach the range, then makes them sparser. | `sparsity_weight` (1), `posthoc_sparsity_param` (0.1) |

`total_CFs` is the number of counterfactuals for each sample. Do not pass `features_to_vary` or `permitted_range`: CELIA sets them from the constraints.

In [ ]:
scenarios = [
    CounterfactualScenario(
        name="random",
        method="dice",
        target=target,
        explainer_kwargs={"method": "random"},
        generate_kwargs={"total_CFs": 3, "random_seed": 0},
    ),
    CounterfactualScenario(
        name="genetic",
        method="dice",
        target=target,
        explainer_kwargs={"method": "genetic"},
        generate_kwargs={"total_CFs": 3},
    ),
    CounterfactualScenario(
        name="kdtree",
        method="dice",
        target=target,
        explainer_kwargs={"method": "kdtree"},
        generate_kwargs={"total_CFs": 3},
    ),
    # The same genetic search, but with a stronger pull towards the sample.
    CounterfactualScenario(
        name="genetic_close",
        method="dice",
        target=target,
        explainer_kwargs={"method": "genetic"},
        generate_kwargs={
            "total_CFs": 3,
            "proximity_weight": 2.0,
            "diversity_weight": 1.0,
        },
    ),
]

### B.3 Evaluate

`samples` are the rows to explain. `reference_input` is the runner input of the reference rows, which DiCE uses to learn the range of each feature.

In [ ]:
report = evaluator.evaluate(
    scenarios,
    samples=X.iloc[:6],
    reference_input={"source": {"X": (X, X_ctx)}},
)
report.summary_frame()[
    [
        "scenario",
        "n_found",
        "n_not_found",
        "n_valid",
        "validity_rate",
        "mean_l1_distance",
        "mean_n_changed",
        "n_violations",
        "total_runtime_s",
    ]
]

How to read the summary:

* `validity_rate`: the share of counterfactuals whose prediction (computed again with the pipeline) is in the target range.
* `mean_l1_distance`: the distance to the sample, over the continuous columns. Each difference is divided by the range of the column in the reference rows. Lower means closer.
* `mean_n_changed`: the number of changed columns. Lower means sparser and easier to act on.
* `n_violations`: changed immutable columns, or values outside the feasible ranges. The evaluator checks this itself, because a method can break a constraint.

The `genetic_close` scenario trades diversity for proximity.

**Why `random` reports violations.** Some samples were sintered below 1 400 °C, outside the feasible range (for example samples 2 and 3). DiCE applies the feasible range only to the values that it changes. `random` changes few columns, so a counterfactual that keeps `sinter_temp_c` also keeps the infeasible value. The evaluator reports this as `feasible:sinter_temp_c`. `genetic` and `kdtree` change more columns and usually move the temperature into the range.

In [ ]:
report.samples_frame()[
    [
        "scenario",
        "sample_id",
        "status",
        "baseline_prediction",
        "target_low",
        "target_high",
        "n_valid",
    ]
]

### B.4 Look at the counterfactuals of one sample

`changes_frame` puts the original row first. `"-"` marks an unchanged value.

In [ ]:
report.changes_frame("random", "0")

In [ ]:
report.changes_frame("genetic_close", "0")

All the counterfactuals, with their metrics, are in `counterfactuals_frame()`. For example, the counterfactuals that change only one column:

In [ ]:
cf = report.counterfactuals_frame()
cf[cf["valid"] & (cf["n_changed"] == 1)][
    ["scenario", "sample_id", "prediction", "changed_columns", "l1_distance"]
]

### B.5 Save the report

The report is a pydantic model: it saves to JSON and loads back with `CounterfactualReport.load_json`.

In [ ]:
import tempfile
from pathlib import Path

path = report.save_json(Path(tempfile.mkdtemp()) / "dice_regression_report.json")
path.name

## Notes

* **Status of a sample.** `found`, `not_found` (DiCE found nothing in the range), `already_in_range` (the baseline is already in the range), or `error` (with a message). One failing sample does not stop the others.
* **Unreachable targets.** If the range is outside what the model can predict, every method returns `not_found`. Check the range of `evaluator.reference_data(...).predictions` first.
* **Reproducibility.** Only `"random"` takes a seed (`random_seed`). `"genetic"` gives slightly different results between runs.
* **Pipelines with CSV sources.** With a `TabularCSVFetcher`, leave `reference_input=None`, and select the samples with `sample_indices=[...]`. If the model predicts several targets, choose one with `OutputPoint(column=...)`. Hide identifier columns with `FeatureConstraints(frozen_columns=[...])`: DiCE is much faster without them.